# Ensemble Retrieval and Fusion for Robust RAG Pipelines

This notebook demonstrates an advanced technique crucial for building highly reliable Retrieval-Augmented Generation (RAG) systems: **Ensemble Retrieval**. While basic RAG pipelines rely on a single retrieval method (e.g., standard cosine similarity), real-world data is complex, and no single search strategy is perfect. Ensemble methods solve this by running multiple diverse retrieval strategies—such as standard similarity search and Maximal Marginal Relevance (MMR)—in parallel. The results are then intelligently fused or weighted to create a comprehensive context block that maximizes both relevance and diversity of information.

For advanced developers working with frameworks like LangGraph, understanding ensemble retrieval is paramount because it directly addresses the "garbage in, garbage out" problem inherent in single-source retrieval. By combining multiple perspectives (e.g., one retriever focusing on direct keyword matches, another prioritizing conceptual diversity), we significantly increase the chances that the final context provided to the Large Language Model (LLM) is comprehensive and non-redundant. This leads to more accurate, grounded, and robust answers, making the entire system less susceptible to failure when faced with ambiguous or complex source material.

By mastering this pattern, you learn how to move beyond simple vector search and build a resilient knowledge retrieval layer. The resulting pipeline—from document loading through multi-retrieval fusion to final LLM generation—is a blueprint for enterprise-grade AI applications that require high fidelity and minimal hallucination risk.

### Learning Objectives
*   **Implement Advanced Document Processing:** Load, chunk, and embed documents using industry-standard tools (PyPDFLoader, RecursiveCharacterTextSplitter).
*   **Understand Vector Store Mechanics:** Initialize and utilize a vector database (Chroma) for efficient semantic search indexing.
*   **Master Diverse Retrieval Strategies:** Implement and compare different retrieval methods, specifically standard similarity search and Maximal Marginal Relevance (MMR), to understand their respective strengths.
*   **Build Ensemble Systems:** Utilize an ensemble mechanism (`RAGFusion`) to fuse the results from multiple retrievers, combining relevance and diversity into a single context block.
*   **Construct Advanced RAG Chains:** Build a complete generation chain using LangChain Expression Language (LCEL) that takes fused context as input, ensuring the LLM is strictly grounded in provided source material.


### RAG Fusion Pipeline - Ensemble Retriever

This notebook demonstrates RAG Fusion using **two retrievers combined via an Ensemble Retriever**.
Instead of generating sub-queries with an LLM, we use two different retrieval strategies on the same vector store and fuse their results using **Reciprocal Rank Fusion (RRF)** built into LangChain's `EnsembleRetriever`.

**Retrievers used:**
- **Similarity Search** - standard cosine similarity retrieval
- **MMR (Maximal Marginal Relevance)** - balances relevance with diversity (`lambda_mult=0.5`)

**Steps covered:**
1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create two retrievers (Similarity + MMR)
5. Apply RAG Fusion via Ensemble Retriever with weights [0.5, 0.5]
6. Augmentation - build context from fused documents
7. Generation - produce a grounded answer using an LLM

### Imports & Setup

### Setup and Initialization

This cell handles the necessary imports for all components of the RAG pipeline (loaders, splitters, embeddings, LLMs, vector stores) and loads environment variables. It ensures that API keys, such as `OPENAI_API_KEY`, are available for subsequent model initialization.


In [2]:
import os
from dotenv import load_dotenv

# Import necessary components for the RAG pipeline
from langchain_community.document_loaders import PyPDFLoader # Used to load PDF documents
from langchain_text_splitters import RecursiveCharacterTextSplitter # Used to split large texts into manageable chunks
from langchain_openai import OpenAIEmbeddings, ChatOpenAI # OpenAI components for embeddings and chat models
from langchain_chroma import Chroma # The vector store implementation
from langchain_core.prompts import ChatPromptTemplate # For defining structured prompts

# Import the custom RAG fusion module (assuming it's defined elsewhere)
from rag_fusion import RAGFusion

# Load OPENAI_API_KEY from the .env file into the environment variables
load_dotenv()


True

### Step 1 - Load the PDF

`PyPDFLoader` reads the PDF and returns one `Document` object per page.

### Documenting Data Loading

This cell initializes a `PyPDFLoader` to read documents from the specified PDF file (`notebooklm_rag.pdf`). The loader processes the document into a list of LangChain `Document` objects, which are then stored in the `pages` variable and counted for verification.


In [3]:
loader = PyPDFLoader("notebooklm_rag.pdf")  # Initialize the PDF loader with the file path
pages = loader.load()  # Load all pages/documents from the PDF into a list

print(f"Loaded {len(pages)} page(s) from the PDF.") # Print confirmation of the number of loaded pages


Loaded 3 page(s) from the PDF.


### Step 2 - Split Documents into Chunks

Large pages are split into smaller, overlapping chunks so that the retriever can surface focused, relevant passages rather than entire pages.

### Document Chunking (Text Splitting)

This cell uses `RecursiveCharacterTextSplitter` to break down large documents (`pages`) into smaller, manageable chunks. This process is crucial for RAG because embedding models have token limits and retrieving small, focused pieces of text improves the relevance and accuracy of the retrieved context.


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split the input documents (pages) into smaller chunks using the defined splitter.
chunks = splitter.split_documents(pages)

# Print the total number of resulting chunks to confirm successful splitting.
print(f"Split into {len(chunks)} chunk(s).")


Split into 19 chunk(s).


### Step 3 - Embeddings & Vector Store

Each chunk is converted into a dense vector using OpenAI's `text-embedding-3-small` model and stored in a ChromaDB vector store.
Both retrievers will query this same vector store using different strategies.

### Vector Store Initialization

This cell initializes and populates a Chroma vector store. It uses the specified `OpenAIEmbeddings` model to convert the document chunks into numerical embeddings, which are then stored in the persistent `Chroma` collection named `notebooklm_rag_ensemble`. This step makes the knowledge base searchable for retrieval-augmented generation (RAG).


In [5]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="notebooklm_rag_ensemble"
)

print("Vector store created successfully.")


Vector store created successfully.


### Step 4 - Create Two Retrievers

We create two retrievers from the same vector store, each using a different search strategy:
- **Similarity Search** - ranks chunks purely by cosine similarity to the query
- **MMR** - Maximal Marginal Relevance reduces redundancy by penalizing chunks that are too similar to already-selected ones. `lambda_mult=0.5` balances relevance and diversity equally.

### Retriever Initialization (Standard Cosine Similarity)

This cell initializes the first retriever component, which uses standard cosine similarity search. The `vectorstore.as_retriever()` method wraps the underlying vector store to provide a dedicated retrieval interface, ensuring it fetches the top $k$ most relevant documents based on semantic similarity.


In [6]:
# Retriever 1: standard cosine similarity search
# We use as_retriever() to wrap the vectorstore object.
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",  # Specifies that we are using standard cosine similarity search.
    search_kwargs={"k": 3}   # Sets 'k' to 3, meaning the retriever will fetch the top 3 most similar documents.
)


### Retriever 2: Maximal Marginal Relevance (MMR)

This cell configures a second retriever using the MMR search type. MMR is crucial for ensuring that the retrieved documents are not only relevant to the query but also diverse, preventing redundancy and providing a broader context for the LLM.


In [7]:
# Retriever 2: MMR - promotes diverse results alongside relevant ones
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr", # Specifies Maximal Marginal Relevance search strategy
    search_kwargs={"k": 3, "lambda_mult": 0.5} # Sets k (number of results) and lambda_mult (diversity weight)
)



### Step 5 - RAG Fusion with Ensemble Retriever

`RAGFusion.from_retrievers` wraps both retrievers inside LangChain's `EnsembleRetriever`.
Equal weights `[0.5, 0.5]` mean both retrievers contribute equally to the final RRF ranking.
When `invoke` is called, the ensemble retriever queries both strategies and merges the results.

### RAG Fusion Ensemble

This cell initializes a `RAGFusion` object, which acts as an ensemble of multiple retrievers (`similarity_retriever`, `mmr_retriever`). By assigning equal weights (0.5) and specifying $k=3$, the system combines the results from both base retrievers to generate a more robust and comprehensive set of top-K documents for the final RAG pipeline.


In [10]:
rag_fusion = RAGFusion.from_retrievers(
    base_retrievers=[similarity_retriever, mmr_retriever],
    weights=[0.5, 0.5],
    k=3
)


### Ensemble Retrieval (RAG Fusion)

This cell demonstrates the core concept of ensemble retrieval. Instead of relying on a single document retriever, it runs multiple specialized retrievers in parallel and uses `rag_fusion` to combine their results using Reciprocal Rank Fusion (RRF). This significantly increases the recall and robustness of the retrieved context.


In [11]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# Both retrievers run in parallel; EnsembleRetriever fuses the results via RRF
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)


Retrieved 3 fused document(s).

--- Document 1 ---
When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

--- Document 2 ---
How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike general-purpose AI assistants, NotebookLM grounds all of its responses in the source

--- Document 3 ---
identifies which specific 

### Step 6 - Augmentation

The retrieved chunks are concatenated into a single context string.
This context will be injected into the generation prompt to ground the LLM's answer.

This cell aggregates the content from all documents stored in `fused_docs` into a single, cohesive string variable named `context`. This is crucial because most LLM prompts require a single block of text for context, and using `

` ensures clear separation between different source chunks.


In [12]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike general-purpose AI assistants, NotebookLM grounds all of its responses in the source

identifies which specific passages from the uploaded documents supported each claim in the answer. These
citations 

### Step 7 - Generation

The context and original query are passed to the LLM via a structured prompt.
The LLM is instructed to answer **only** from the provided context and to say `"I don't know"` if the answer isn't there.

### Contextual Generation Chain

This cell defines and executes a basic Retrieval-Augmented Generation (RAG) chain. It uses `ChatPromptTemplate` to structure the input, ensuring the LLM is constrained to use only provided context (`{context}`) when answering the `{question}`. The resulting `generation_chain` pipes the structured prompt into the specified OpenAI model.


In [13]:
llm = ChatOpenAI(model="gpt-5-mini")

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

It first automatically indexes the uploaded file: extracting the raw text (using OCR for scanned PDF pages or direct extraction for digital PDFs), then cleaning and normalizing that text. The text is split into overlapping chunks with a strategy that preserves semantic coherence. At query time NotebookLM retrieves the relevant chunks/passages from that indexed text, uses them to support its answers, and surfaces the specific source passages as inline citations. If the retrieved context is insufficient, it explicitly acknowledges that limitation.
